# Exercise: Hierarchical models and testing
**Clara Hardes, 22.09.2026** <br>

This analysis investigates whether descriptive social norms influence hotel guests' behaviour regarding towel reuse. The dataset contains several randomized case-control studies comparing a control group with a social-norm intervention group. Following Scheibehenne et al. (2016), we use a Bayesian hierarchical model to combine the evidence across studies while accounting for differences between studies.

The outcome is modelled as a binomial response, with the number of guests reusing their towels out of the total number of guests in each group. The model allows both baseline towel-reuse rates and intervention effects to vary between studies. We estimate the model parameters using Bayesian inference and evaluate whether the posterior evidence supports a positive effect of the social-norm intervention on towel reuse.

In [20]:
#import packages
import pandas as pd
import numpy as np
import arviz as az
import bambi as bmb

## Data preparation

Code from Ulrika's github.

In [21]:
# read in data
data_file = r"towelData.csv"
data = pd.read_csv(data_file, sep=';', encoding='latin1')
count = data.iloc[:, -1] # get the last column with numbers or yes/no

# count has the number of yes and no for control and social norm groups
control_yes = count[::4].to_numpy() # every 4th starting from 0 - control group + yes
control_no = count[2::4].to_numpy() # every 4th starting from 2 - control group + no
control_total = np.array([y + n for y, n in zip(control_yes, control_no)])

social_yes = count[1::4].to_numpy() # every 4th starting from 1 - social norm group + yes
social_no = count[3::4].to_numpy() # every 4th starting from 3 - social norm group + no
social_total = np.array([y + n for y, n in zip(social_yes, social_no)])

study = np.arange(1,len(control_yes)+1) # 7 diferent studies

control_data = pd.DataFrame({"reuse": control_yes, "total": control_total, "group": "control", "study": study})
social_data = pd.DataFrame({ "reuse": social_yes, "total": social_total, "group": "social", "study": study})

combined_data = pd.concat([control_data, social_data], ignore_index=True)

# Convert data types - important for bambi that they are correctly set
# can give errors otherwise
combined_data['reuse'] = combined_data['reuse'].astype(int)
combined_data['total'] = combined_data['total'].astype(int)
combined_data['group'] = combined_data['group'].astype('category')
combined_data['study'] = combined_data['study'].astype('category')

In [12]:
combined_data

,reuse,total,group,study
0,74,211,control,1
1,103,277,control,2
2,77,135,control,3
3,82,187,control,4
4,21,25,control,5
5,123,147,control,6
6,28,30,control,7
7,98,222,social,1
8,587,1318,social,2
9,406,655,social,3


## Model description
To estimate the model parameters the function bmb.Model from bambi is used.The model models towel reuse using a binomial response.

```python
p(reuse, total) ~ group + (1 + group | study)
```

`group` represents the difference between the social norm intervention effect (social/control). The hierarchical term `(1 + group | study)` allows both the baseline reuse rate and the intervention effect to vary between studies. Thus, the model accounts for between-study heterogeneity in both baseline behaviour and the effectiveness of the intervention (the intercept and the slope).


In [11]:
#set up model using bambi
model = bmb.Model(
    "p(reuse, total) ~ group + (1 + group | study)",
    data=combined_data,
    family="binomial"
)

print("\nModel:")
print(model)

# Fit the Bayesian model, set target_accept to 0.95 to make it faster
idata = model.fit(
    target_accept = 0.95
)



Model:
       Formula: p(reuse, total) ~ group + (1 + group | study)
        Family: binomial
          Link: p = logit
  Observations: 14
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 1.5)
            group ~ Normal(mu: 0.0, sigma: 1.0)
        
        Group-level effects
            1|study ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5495))
            group|study ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 5.0))


Output()

## Hypotheses
**H_0**: The descriptive social norms intervention does not increase towel reuse.<br>

**H_1**: The descriptive social norms intervention increases towel reuse.

$$
H_0: \beta_{\text{group}} \leq 0
$$

$$
H_1: \beta_{\text{group}} > 0
$$

$\beta_{\text{group}}$  = the model parameter of difference between the groups.

To test the hypotheses we first look at the overall summary of the model and then calculate the posterior probability for $\beta_{\text{group}}> 0 | data$.

In [13]:
#print the overall summary of the model
summary = az.summary(
    idata,
    ci_prob=0.90,
    kind="stats"
)

print(summary)

                             mean    sd eti90_lb eti90_ub
Intercept                    0.43  0.44    -0.28      1.1
group[social]                0.18  0.15   -0.058     0.38
1|study_sigma                 1.2  0.42     0.67      1.9
group|study_sigma[social]     0.2   0.2    0.014     0.59
1|study[1]                  -0.97  0.45     -1.7    -0.24
1|study[2]                   -0.9  0.45     -1.6    -0.15
1|study[3]                  -0.14  0.45    -0.85     0.57
1|study[4]                  -0.64  0.45     -1.4    0.095
1|study[5]                    1.2  0.55     0.32      2.1
1|study[6]                      1  0.45      0.3      1.8
1|study[7]                   0.91  0.52     0.12      1.8
group|study[social, 1]      0.086  0.18    -0.13     0.45
group|study[social, 2]      0.067  0.16    -0.13     0.38
group|study[social, 3]      0.021  0.16    -0.22      0.3
group|study[social, 4]      0.038  0.16    -0.17     0.32
group|study[social, 5]      0.034  0.24    -0.27     0.42
group|study[so

In [15]:
# get the name of the variables
print(idata.posterior.data_vars)

Data variables:
    Intercept          (chain, draw) float64 16kB 0.5471 0.3903 ... 0.03514
    group              (chain, draw, group_dim) float64 16kB 0.2061 ... 0.05618
    1|study_sigma      (chain, draw) float64 16kB 1.242 1.535 ... 0.4261 1.443
    group|study_sigma  (chain, draw, group__expr_dim) float64 16kB 0.05261 .....
    1|study            (chain, draw, study__factor_dim) float64 112kB -1.037 ...
    group|study        (chain, draw, group__expr_dim, study__factor_dim) float64 112kB ...


In [18]:
#calculate P(beta_group > 0 | data)
beta = idata.posterior["group"]

prob_positive = (beta > 0).mean().item()

print(f"P(beta_group > 0 | data) = {prob_positive:.3f}")

P(beta_group > 0 | data) = 0.907


## Conclusion
The hierarchical model allows both the baseline reuse rate and the intervention effect to vary between studies. The posterior mean of the overall intervention effect is $\beta_{\text{group}} = 0.18$, with a 90% posterior interval of $[-0.058, 0.38]$. The posterior probability of a positive intervention effect is
$$
P(\beta_{\text{group}}>0\mid\text{data})=0.907.
$$
Therefore, there is a 90.7% posterior probability that the social norm intervention increases towel reuse. However, The 90% posterior interval includes zero, indicating uncertainty about the size of the effect.

The estimated between-study standard deviation of the intervention effect is $0.20$, with a 90% posterior interval of $[0.014, 0.59]$, indicating heterogenity across the studies. Thus it indicates variations in the effectiveness of the social norm intervention across different studies. However, none of the individual study-specific slopes has a 90% posterior interval entirely above zero, therefore in all studies there is uncertainty regarding the magnitude of effect of the intervention.